# Slot-Builder v2 Repair Corpus Notebook

Trunk only:

```text
slot_builder_lora_v1_eval
  ↓
Ω residue examples
  ↓
repair rows
  ↓
slot_builder_lora_v2 corpus
```

Run this from **Downloads**.

Inputs expected beside the notebook:

```text
slot_builder_lora_v1_eval/slot_builder_eval_records.jsonl
slot_builder_lora_v1_eval/slot_builder_eval_summary.csv
nexus_ai_shape_pull_outputs/rhi_contract_loop_examples.jsonl
v73b_outputs_local_root_prompt_recovered_slot_corpus/slot_sft_messages.jsonl
```

Output:

```text
slot_builder_v2_repair_corpus/
  slot_repair_training_pairs.jsonl
  slot_repair_sft_messages.jsonl
  slot_repair_preference_pairs.jsonl
  slot_builder_v2_combined_sft_messages.jsonl
  slot_repair_training_pairs.csv
  slot_repair_audit.csv
  slot_repair_manifest.json
```

Primary v2 training file:

```text
slot_builder_v2_repair_corpus/slot_builder_v2_combined_sft_messages.jsonl
```


In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

EVAL_DIR = ROOT / "slot_builder_lora_v1_eval"
EVAL_RECORDS_FILE = EVAL_DIR / "slot_builder_eval_records.jsonl"
EVAL_SUMMARY_FILE = EVAL_DIR / "slot_builder_eval_summary.csv"

SHAPE_DIR = ROOT / "nexus_ai_shape_pull_outputs"
RESIDUE_EXAMPLES_FILE = SHAPE_DIR / "rhi_contract_loop_examples.jsonl"

V73B_DIR = ROOT / "v73b_outputs_local_root_prompt_recovered_slot_corpus"
BASE_SFT_FILE = V73B_DIR / "slot_sft_messages.jsonl"

OUTPUT_DIR = ROOT / "slot_builder_v2_repair_corpus"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INCLUDE_CLEANUP_FIXES = True
INCLUDE_FAILED_CLEANED = True
INCLUDE_EXACT_STABILIZERS = False

print("ROOT:", ROOT)
print("EVAL_RECORDS_FILE:", EVAL_RECORDS_FILE, EVAL_RECORDS_FILE.exists())
print("RESIDUE_EXAMPLES_FILE:", RESIDUE_EXAMPLES_FILE, RESIDUE_EXAMPLES_FILE.exists())
print("BASE_SFT_FILE:", BASE_SFT_FILE, BASE_SFT_FILE.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# ============================================================
# IMPORTS AND HELPERS
# ============================================================
import json
import re
from typing import Any, Dict, List, Optional
import pandas as pd

REQUIRED_FIELDS = [
    "family_class",
    "domain_carrier",
    "forbidden_neighbor_carrier",
    "boundary_conditions",
    "preserved_function",
    "failure_modes",
    "witness_readout",
    "residue",
]

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows: List[Dict[str, Any]]):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def extract_first_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = str(text).strip()
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    cleaned = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        obj = json.loads(cleaned)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end+1])
            return obj if isinstance(obj, dict) else None
        except Exception:
            return None
    return None

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        if x.strip().startswith("{") or ":" in x:
            return [x.strip()]
        return [p.strip() for p in re.split(r"[|,;]", x) if p.strip()]
    return [str(x).strip()]

def normalize_contract(c0: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    c0 = c0 or {}
    return {
        "family_class": str(c0.get("family_class", "") or "").strip(),
        "domain_carrier": normalize_list(c0.get("domain_carrier", [])),
        "forbidden_neighbor_carrier": normalize_list(c0.get("forbidden_neighbor_carrier", [])),
        "boundary_conditions": normalize_list(c0.get("boundary_conditions", [])),
        "preserved_function": str(c0.get("preserved_function", "") or "").strip(),
        "failure_modes": normalize_list(c0.get("failure_modes", [])),
        "witness_readout": str(c0.get("witness_readout", "") or "").strip(),
        "residue": c0.get("residue", None),
    }

def contract_complete(contract: Optional[Dict[str, Any]]) -> bool:
    if not isinstance(contract, dict):
        return False
    contract = normalize_contract(contract)
    for field in REQUIRED_FIELDS:
        if field == "residue":
            continue
        value = contract.get(field)
        if isinstance(value, list):
            if len(value) == 0:
                return False
        elif not str(value or "").strip():
            return False
    return True

def diff_contract(current: Dict[str, Any], target: Dict[str, Any]) -> Dict[str, Any]:
    current = normalize_contract(current)
    target = normalize_contract(target)
    mismatched = []
    field_diff = {}
    for field in REQUIRED_FIELDS:
        if current.get(field) != target.get(field):
            mismatched.append(field)
            field_diff[field] = {"current": current.get(field), "target": target.get(field)}
    return {
        "mismatched_fields": mismatched,
        "field_diff": field_diff,
        "n_mismatched_fields": len(mismatched),
    }

print("helpers ready")


In [ ]:
# ============================================================
# LOAD INPUTS
# ============================================================
if not EVAL_RECORDS_FILE.exists():
    raise FileNotFoundError("Missing eval records: " + str(EVAL_RECORDS_FILE))

if not BASE_SFT_FILE.exists():
    raise FileNotFoundError("Missing base SFT corpus: " + str(BASE_SFT_FILE))

eval_records = read_jsonl(EVAL_RECORDS_FILE)
base_sft_rows = read_jsonl(BASE_SFT_FILE)
residue_examples = read_jsonl(RESIDUE_EXAMPLES_FILE)

eval_summary_df = pd.read_csv(EVAL_SUMMARY_FILE) if EVAL_SUMMARY_FILE.exists() else pd.DataFrame()

sft_by_id = {row.get("base_id"): row for row in base_sft_rows if row.get("base_id")}
residue_by_id = {row.get("case_id"): row for row in residue_examples if row.get("case_id")}

print("eval_records:", len(eval_records))
print("base_sft_rows:", len(base_sft_rows))
print("residue_examples:", len(residue_examples))
print("sft_by_id:", len(sft_by_id))
print("residue_by_id:", len(residue_by_id))


In [ ]:
# ============================================================
# EXTRACT ORIGINAL PROMPTS AND TARGET CONTRACTS
# ============================================================
def get_original_prompt_from_sft(row: Dict[str, Any]) -> str:
    messages = row.get("messages", [])
    if len(messages) < 2:
        return ""

    user_content = messages[1].get("content", "")
    if "Prompt:" in user_content:
        after = user_content.split("Prompt:", 1)[1].strip()
        stop = "\n\nGenerate the missing-shape contract."
        if stop in after:
            return after.split(stop, 1)[0].strip()
        return after.strip()
    return user_content.strip()

def get_target_contract_from_sft(row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    messages = row.get("messages", [])
    if not messages:
        return None
    return extract_first_json_object(messages[-1].get("content", ""))

prompt_by_id = {}
target_by_id = {}

for base_id, row in sft_by_id.items():
    prompt_by_id[base_id] = get_original_prompt_from_sft(row)
    target_by_id[base_id] = normalize_contract(get_target_contract_from_sft(row))

print("prompt_by_id:", len(prompt_by_id))
print("target_by_id:", len(target_by_id))
print("sample prompt:", next(iter(prompt_by_id.values()))[:240])


In [ ]:
# ============================================================
# BUILD REPAIR ROWS
# ============================================================
SYSTEM_REPAIR = '''You are the Nexus Slot Repairer.

Your job is to repair a malformed missing-shape contract.

Do not answer the task.
Do not mention answer choices.
Return strict JSON only.

Use the prompt, the current contract, the residue, and the field diff.
The output must be the repaired contract.
'''

def make_repair_user_message(prompt, current_contract, residue, field_diff):
    return (
        "Original prompt:\n"
        + prompt
        + "\n\nCurrent malformed contract:\n"
        + json.dumps(normalize_contract(current_contract), ensure_ascii=False, indent=2)
        + "\n\nΩ residue:\n"
        + json.dumps(residue, ensure_ascii=False, indent=2)
        + "\n\nField diff:\n"
        + json.dumps(field_diff, ensure_ascii=False, indent=2)
        + "\n\nRepair the contract only. Return strict JSON only."
    )

def make_messages(prompt, current_contract, residue, field_diff, target_contract):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_REPAIR},
            {"role": "user", "content": make_repair_user_message(prompt, current_contract, residue, field_diff)},
            {"role": "assistant", "content": json.dumps(normalize_contract(target_contract), ensure_ascii=False, indent=2)},
        ]
    }

repair_pairs = []
repair_sft_messages = []
repair_preference_pairs = []

for rec in eval_records:
    base_id = rec.get("base_id")
    if not base_id:
        continue

    prompt = prompt_by_id.get(base_id, "")
    if not prompt:
        continue

    generated = normalize_contract(rec.get("generated_contract"))
    cleaned = normalize_contract(rec.get("cleaned_contract"))
    expected = normalize_contract(rec.get("expected_contract") or target_by_id.get(base_id))

    if not contract_complete(expected):
        continue

    exact_generated = bool(rec.get("exact_generated", False))
    exact_cleaned = bool(rec.get("exact_cleaned", False))

    include = False
    repair_kind = None
    current_contract = None

    if INCLUDE_CLEANUP_FIXES and (not exact_generated) and exact_cleaned:
        include = True
        repair_kind = "cleanup_fixed_generated_to_target"
        current_contract = generated
    elif INCLUDE_FAILED_CLEANED and (not exact_cleaned):
        include = True
        repair_kind = "cleaned_still_failed_to_target"
        current_contract = cleaned
    elif INCLUDE_EXACT_STABILIZERS and exact_cleaned:
        include = True
        repair_kind = "exact_stabilizer"
        current_contract = cleaned

    if not include:
        continue

    field_diff = diff_contract(current_contract, expected)
    known = residue_by_id.get(base_id, {})

    residue = {
        "case_id": base_id,
        "repair_kind": repair_kind,
        "known_omega": known.get("omega"),
        "known_symptom": known.get("symptom"),
        "known_repair": known.get("repair"),
        "mismatched_fields": field_diff["mismatched_fields"],
        "n_mismatched_fields": field_diff["n_mismatched_fields"],
        "exact_generated": exact_generated,
        "exact_cleaned": exact_cleaned,
    }

    pair = {
        "base_id": base_id,
        "prompt": prompt,
        "repair_kind": repair_kind,
        "current_contract": normalize_contract(current_contract),
        "target_contract": normalize_contract(expected),
        "residue": residue,
        "field_diff": field_diff,
    }
    repair_pairs.append(pair)

    msg = make_messages(prompt, current_contract, residue, field_diff, expected)
    msg["base_id"] = base_id
    msg["repair_kind"] = repair_kind
    msg["training_family"] = "slot_repair"
    repair_sft_messages.append(msg)

    repair_preference_pairs.append({
        "base_id": base_id,
        "prompt": make_repair_user_message(prompt, current_contract, residue, field_diff),
        "chosen": json.dumps(normalize_contract(expected), ensure_ascii=False, indent=2),
        "rejected": json.dumps(normalize_contract(current_contract), ensure_ascii=False, indent=2),
        "metadata": {"repair_kind": repair_kind, "residue": residue, "field_diff": field_diff},
    })

print("repair_pairs:", len(repair_pairs))
display(pd.DataFrame([{
    "base_id": r["base_id"],
    "repair_kind": r["repair_kind"],
    "n_mismatched_fields": r["field_diff"]["n_mismatched_fields"],
    "mismatched_fields": " | ".join(r["field_diff"]["mismatched_fields"]),
    "known_omega": r["residue"]["known_omega"],
} for r in repair_pairs]))


In [ ]:
# ============================================================
# COMBINE ORIGINAL SFT + REPAIR SFT FOR v2
# ============================================================
combined_sft_messages = []

for row in base_sft_rows:
    out = dict(row)
    out["training_family"] = "slot_builder_original"
    combined_sft_messages.append(out)

combined_sft_messages.extend(repair_sft_messages)

print("original SFT:", len(base_sft_rows))
print("repair SFT:", len(repair_sft_messages))
print("combined v2 SFT:", len(combined_sft_messages))


In [ ]:
# ============================================================
# AUDIT AND SAVE
# ============================================================
audit_rows = []
for pair in repair_pairs:
    audit_rows.append({
        "base_id": pair["base_id"],
        "repair_kind": pair["repair_kind"],
        "prompt_present": bool(pair["prompt"].strip()),
        "current_complete": contract_complete(pair["current_contract"]),
        "target_complete": contract_complete(pair["target_contract"]),
        "n_mismatched_fields": pair["field_diff"]["n_mismatched_fields"],
        "mismatched_fields": " | ".join(pair["field_diff"]["mismatched_fields"]),
        "known_omega": pair["residue"].get("known_omega"),
        "known_repair": pair["residue"].get("known_repair"),
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

ready = bool(
    len(repair_pairs) > 0
    and audit_df["prompt_present"].all()
    and audit_df["current_complete"].all()
    and audit_df["target_complete"].all()
)

write_jsonl(OUTPUT_DIR / "slot_repair_training_pairs.jsonl", repair_pairs)
write_jsonl(OUTPUT_DIR / "slot_repair_sft_messages.jsonl", repair_sft_messages)
write_jsonl(OUTPUT_DIR / "slot_repair_preference_pairs.jsonl", repair_preference_pairs)
write_jsonl(OUTPUT_DIR / "slot_builder_v2_combined_sft_messages.jsonl", combined_sft_messages)

csv_rows = []
for pair in repair_pairs:
    csv_rows.append({
        "base_id": pair["base_id"],
        "repair_kind": pair["repair_kind"],
        "prompt": pair["prompt"],
        "known_omega": pair["residue"].get("known_omega"),
        "known_symptom": pair["residue"].get("known_symptom"),
        "known_repair": pair["residue"].get("known_repair"),
        "n_mismatched_fields": pair["field_diff"]["n_mismatched_fields"],
        "mismatched_fields": " | ".join(pair["field_diff"]["mismatched_fields"]),
        "current_family_class": pair["current_contract"].get("family_class"),
        "target_family_class": pair["target_contract"].get("family_class"),
        "current_domain_carrier": " | ".join(pair["current_contract"].get("domain_carrier", [])),
        "target_domain_carrier": " | ".join(pair["target_contract"].get("domain_carrier", [])),
    })

repair_csv_df = pd.DataFrame(csv_rows)
repair_csv_df.to_csv(OUTPUT_DIR / "slot_repair_training_pairs.csv", index=False)
audit_df.to_csv(OUTPUT_DIR / "slot_repair_audit.csv", index=False)

manifest = {
    "notebook": "slot_builder_v2_repair_corpus_notebook",
    "root": str(ROOT),
    "output_dir": str(OUTPUT_DIR),
    "inputs": {
        "eval_records": str(EVAL_RECORDS_FILE),
        "eval_summary": str(EVAL_SUMMARY_FILE),
        "residue_examples": str(RESIDUE_EXAMPLES_FILE),
        "base_sft": str(BASE_SFT_FILE),
    },
    "counts": {
        "eval_records": len(eval_records),
        "base_sft_rows": len(base_sft_rows),
        "residue_examples": len(residue_examples),
        "repair_pairs": len(repair_pairs),
        "repair_sft_messages": len(repair_sft_messages),
        "repair_preference_pairs": len(repair_preference_pairs),
        "combined_sft_messages": len(combined_sft_messages),
    },
    "ready": {
        "repair_corpus_ready": ready,
        "v2_combined_sft_ready": bool(ready and len(combined_sft_messages) > len(base_sft_rows)),
    },
    "primary_training_file": "slot_builder_v2_combined_sft_messages.jsonl",
}

(OUTPUT_DIR / "slot_repair_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))
print()
print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


# Ψ readout

Primary v2 training file:

```text
slot_builder_v2_repair_corpus/slot_builder_v2_combined_sft_messages.jsonl
```

It contains:

```text
original Q → C rows
+
repair Ω → C' rows
```

Training target for v2:

$$
(Q \rightarrow C)
\oplus
(Q,C_{bad},\Omega,\Delta C \rightarrow C')
$$

Next trunk step after this notebook runs:

```text
train slot_builder_lora_v2 on slot_builder_v2_combined_sft_messages.jsonl
```
